In [1]:
%%time
import os

# Must be set BEFORE importing torch / transformers in a fresh kernel.
# These physical GPU IDs become logical cuda:0 and cuda:1 inside this kernel.
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

# The GPUs visible inside this kernel are now:
# cuda:0 -> physical GPU 4
# cuda:1 -> physical GPU 5
print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}")

# Choose ONE model per execution:
model_name = "Qwen/Qwen2.5-7B-Instruct"
# model_name = "google/gemma-3-4b-it"

# Leave headroom for the KV cache, input tensors, and temporary allocations.
# Your A16 cards have approximately 14.61 GiB usable VRAM each.
max_memory = {
    0: "12GiB",
    1: "12GiB",
    "cpu": "100GiB",
}

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="balanced",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

model.eval()

print("\nModel device map:")
print(model.hf_device_map)

# With a multi-GPU device map, inputs belong on the model's input device.
# For this configuration, it is normally cuda:0.
input_device = next(model.parameters()).device
print("\nInput device:", input_device)

prompt = "Give me a short introduction to large language models."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer(
    [text],
    return_tensors="pt",
).to(input_device)

print("input_ids device:", model_inputs["input_ids"].device)

with torch.inference_mode():
    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False,              # Deterministic and better for timing
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

new_tokens = output_ids[:, model_inputs["input_ids"].shape[1]:]
response = tokenizer.batch_decode(
    new_tokens,
    skip_special_tokens=True,
)[0]

print("\nResponse:\n", response)

/var/lib/datausers_jupyterhub/lfalconi_storage/miniforge3/envs/pyt-eqa-fge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Visible GPUs: 2
cuda:0: NVIDIA A16
cuda:1: NVIDIA A16


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 102.31it/s]



Model device map:
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}

Input device: cuda:0
input_ids device: cuda:0

Response:
 Large language models (LLMs) are artificial intelligence systems designed to understand and generate human-like text based on the input they receive. These models are typically trained on vast amounts of textual data from the internet, 

In [2]:
new_tokens

tensor([[ 34253,   4128,   4119,    320,   4086,  21634,      8,    525,  20443,
          11229,   5942,   6188,    311,   3535,    323,   6923,   3738,  12681,
           1467,   3118,    389,    279,   1946,    807,   5258,     13,   4220,
           4119,    525,  11136,  16176,    389,  12767,  14713,    315,  62533,
            821,    504,    279,   7602,     11,   6467,     11,    323,   1008,
           8173,     11,  10693,   1105,    311,   3960,  12624,     11,  37597,
             11,    323,  83789,    304,   4128,     13,    444,  10994,     82,
            646,   2736,    264,   6884,   2088,    315,   5810,   4128,   8692,
           9079,     11,   1741,    438,  14468,     11,  28285,   2022,     11,
           3405,     12,    596,     86,   4671,     11,    323,   1496,  11521,
           4378,    382,   3966,    315,    279,   1376,   4419,    315,    444,
          10994,     82,    374,    862,   1379,     26,    807,   3545,   6644,
          32051,    476,   1

In [3]:
def gpu_memory(device_id):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3
    total = torch.cuda.get_device_properties(device_id).total_memory / 1024**3
    print(
        f"cuda:{device_id} | "
        f"allocated: {allocated:.2f} GiB | "
        f"reserved: {reserved:.2f} GiB | "
        f"total: {total:.2f} GiB"
    )

gpu_memory(0)
gpu_memory(1)

cuda:0 | allocated: 6.23 GiB | reserved: 6.26 GiB | total: 14.61 GiB
cuda:1 | allocated: 7.97 GiB | reserved: 7.99 GiB | total: 14.61 GiB


In [4]:
import gc
import torch

# Remove Python references to GPU-resident objects
for name in (
    "model",
    "tokenizer",
    "model_inputs",
    "generated_ids",
    "response",
    "text",
    "messages",
):
    globals().pop(name, None)

# Force Python garbage collection, then release unused PyTorch cache
gc.collect()

for device_id in range(torch.cuda.device_count()):
    with torch.cuda.device(device_id):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()